# Регрессия SI

Для `SI` важно сравнить два сценария: прямой прогноз по дескрипторам и косвенный расчёт через `CC50 / IC50`.

Это самая капризная регрессионная задача: хвост тяжёлый, а косвенный подход легко становится нестабильным.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Рабочая директория: {PROJECT_ROOT}')

Рабочая директория: /Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry


In [2]:
from types import SimpleNamespace

import pandas as pd
from IPython.display import Image, display

from src.common.config import RESULTS_DIR, TASKS
from src.common.data import find_dataset_path, load_dataset
from src.common.preprocessing import prepare_task_data
from src.common.training import run_regression_si_task, run_supervised_task

## Настройки запуска

По умолчанию стоит полный режим `nested`. Если нужен быстрый прогон для проверки,
достаточно переключить `EVALUATION_STRATEGY` на `"holdout"`.

In [3]:
TASK_NAME = 'regression_si'
EVALUATION_STRATEGY = 'nested'
MODELS = None
SKIP_CATBOOST = False
OUTER_FOLDS = 5
INNER_FOLDS = 3
TEST_SIZE = 0.2
RANDOM_SEED = 42
TOP_K_IMPORTANCE = 20

task = TASKS[TASK_NAME]
dataframe = load_dataset(find_dataset_path())
prepared = prepare_task_data(dataframe, task)

print(f'Задача: {task.title}')
print(f'Матрица признаков: {prepared["X"].shape}')
print(f'Число признаков после фильтрации: {len(prepared["feature_columns"])}')
print(f'Статус проверки на утечку: {prepared["leakage_report"]["status"]}')

Задача: Регрессия: SI
Матрица признаков: (1001, 210)
Число признаков после фильтрации: 210
Статус проверки на утечку: passed


## Быстрый срез по таргету

Перед обучением полезно один раз посмотреть, что именно мы подаём в модель.

In [4]:
if task.problem_type == 'classification':
    target_frame = (
        prepared['y']
        .value_counts()
        .sort_index()
        .rename_axis('label')
        .reset_index(name='count')
    )
    target_frame['share'] = target_frame['count'] / target_frame['count'].sum()
    display(target_frame)
else:
    display(prepared['y'].describe().to_frame(name=task.target_column).T)

,count,mean,std,min,25%,50%,75%,max
SI,1001.0,72.508823,684.482739,0.011489,1.433333,3.846154,16.566667,15620.6


## Запуск эксперимента

Эта ячейка пересчитывает результаты, пишет артефакты в `results/` и обновляет текстовый отчёт в `reports/`.

In [5]:
args = SimpleNamespace(
    evaluation_strategy=EVALUATION_STRATEGY,
    models=MODELS,
    skip_catboost=SKIP_CATBOOST,
    outer_folds=OUTER_FOLDS,
    inner_folds=INNER_FOLDS,
    test_size=TEST_SIZE,
    random_seed=RANDOM_SEED,
    top_k_importance=TOP_K_IMPORTANCE,
)

summary = run_regression_si_task(task, args)
summary

2026-04-20 17:53:17,852 | INFO | Running SI direct vs indirect with models: ['dummy', 'ridge', 'lasso', 'knn', 'svr', 'random_forest', 'extra_trees', 'gradient_boosting', 'catboost']
2026-04-20 17:53:17,853 | INFO | Evaluating direct SI model dummy
2026-04-20 17:53:18,321 | INFO | Evaluating indirect SI model dummy
2026-04-20 17:53:19,230 | INFO | Evaluating direct SI model ridge
2026-04-20 17:53:20,225 | INFO | Evaluating indirect SI model ridge
2026-04-20 17:53:22,178 | INFO | Evaluating direct SI model lasso
/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/.venv/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.485e-01, tolerance: 1.171e-01
  model = cd_fast.enet_coordinate_descent(
/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural N

{'task_name': 'regression_si',
 'title': 'Регрессия: SI',
 'problem_type': 'regression',
 'target_column': 'SI',
 'primary_metric': 'mae',
 'evaluation_strategy': 'nested',
 'random_seed': 42,
 'leaderboard_path': '/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/results/regression_si/leaderboard.csv',
 'data_contract_path': '/Users/davidsukhashvili/Desktop/ML/MEPhi/ClassicMl/Neural Networks in Chemistry/results/data_contract.json',
 'best_direct': {'task_name': 'regression_si',
  'problem_type': 'regression',
  'target_column': 'SI',
  'primary_metric': 'mae',
  'evaluation_strategy': 'nested',
  'model_name': 'catboost',
  'mode': 'direct',
  'fit_seconds': 795.6792905839975,
  'best_params_json': '{"depth": 6, "iterations": 400, "l2_leaf_reg": 1, "learning_rate": 0.03}',
  'mae': 67.52714787753031,
  'mae_std': 57.633044352273764,
  'rmse': 476.62009529312354,
  'rmse_std': 485.4333165391032,
  'r2': 0.03206840075250095,
  'r2_std': 0.02326816573662733

## Лидерборд

Здесь уже удобно смотреть не только на победителя, но и на разрыв с бейзлайном.

In [6]:
leaderboard = pd.read_csv(RESULTS_DIR / 'regression_si' / 'leaderboard.csv')
display(leaderboard)

,task_name,problem_type,target_column,primary_metric,evaluation_strategy,model_name,mode,fit_seconds,best_params_json,mae,mae_std,rmse,rmse_std,r2,r2_std
0,regression_si,regression,SI,mae,nested,extra_trees,indirect,219.480868,"{""cc50_component"": {""max_depth"": null, ""max_fe...",6.729125e+01,5.735437e+01,4.785548e+02,4.856427e+02,2.206342e-02,2.567825e-02
1,regression_si,regression,SI,mae,nested,catboost,direct,795.679291,"{""depth"": 6, ""iterations"": 400, ""l2_leaf_reg"":...",6.752715e+01,5.763304e+01,4.766201e+02,4.854333e+02,3.206840e-02,2.326817e-02
2,regression_si,regression,SI,mae,nested,random_forest,indirect,475.730031,"{""cc50_component"": {""max_depth"": null, ""max_fe...",6.770791e+01,5.753012e+01,4.781361e+02,4.861901e+02,2.918784e-02,3.664024e-02
3,regression_si,regression,SI,mae,nested,extra_trees,direct,111.338981,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m...",6.792080e+01,5.767094e+01,4.758927e+02,4.845787e+02,2.819184e-02,2.748291e-02
4,regression_si,regression,SI,mae,nested,catboost,indirect,1601.896838,"{""cc50_component"": {""depth"": 6, ""iterations"": ...",6.797636e+01,5.757602e+01,4.799852e+02,4.862257e+02,1.702360e-02,3.141417e-02
5,regression_si,regression,SI,mae,nested,random_forest,direct,238.516946,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m...",6.801638e+01,5.737034e+01,4.763791e+02,4.839952e+02,2.392152e-02,2.594594e-02
6,regression_si,regression,SI,mae,nested,gradient_boosting,direct,235.692208,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",6.805213e+01,5.755456e+01,4.775822e+02,4.855745e+02,2.214243e-02,1.850566e-02
7,regression_si,regression,SI,mae,nested,gradient_boosting,indirect,457.094882,"{""cc50_component"": {""learning_rate"": 0.1, ""max...",6.879816e+01,5.757303e+01,4.800713e+02,4.868931e+02,2.010994e-02,3.646404e-02
8,regression_si,regression,SI,mae,nested,svr,direct,11.414912,"{""C"": 1.0, ""epsilon"": 0.01, ""gamma"": ""scale""}",6.891308e+01,5.719210e+01,4.816161e+02,4.857306e+02,-4.109368e-03,7.738805e-03
9,regression_si,regression,SI,mae,nested,knn,direct,5.444663,"{""n_neighbors"": 21, ""p"": 1, ""weights"": ""distan...",6.937777e+01,5.825395e+01,4.780774e+02,4.853310e+02,2.431047e-02,3.733226e-02


## Короткий разбор результата

In [7]:
winner = leaderboard.iloc[0]
baseline_rows = leaderboard[leaderboard['model_name'] == 'dummy']
baseline = baseline_rows.iloc[0] if not baseline_rows.empty else None

print(
    f"Победитель: {winner['model_name']} "
    f"({winner['mode']}), "
    f"основная метрика {winner['primary_metric']} = {winner[winner['primary_metric']]:.6f}."
)
if baseline is not None:
    print(
        f"Для сравнения dummy даёт {baseline[baseline['primary_metric']]:.6f} по той же метрике."
    )
print(f"Лучшие параметры: {winner['best_params_json']}")

Победитель: extra_trees (indirect), основная метрика mae = 67.291247.
Для сравнения dummy даёт 70.674322 по той же метрике.
Лучшие параметры: {"cc50_component": {"max_depth": null, "max_features": 0.5, "min_samples_leaf": 1}, "ic50_component": {"max_depth": 16, "max_features": 0.5, "min_samples_leaf": 5}}


## Прямой против косвенного подхода

Для `SI` мало просто выбрать одну модель. Здесь важнее понять, какой сам сценарий устойчивее.

In [8]:
direct_rows = leaderboard[leaderboard['mode'] == 'direct'].reset_index(drop=True)
indirect_rows = leaderboard[leaderboard['mode'] == 'indirect'].reset_index(drop=True)

display(direct_rows)
display(indirect_rows)

,task_name,problem_type,target_column,primary_metric,evaluation_strategy,model_name,mode,fit_seconds,best_params_json,mae,mae_std,rmse,rmse_std,r2,r2_std
0,regression_si,regression,SI,mae,nested,catboost,direct,795.679291,"{""depth"": 6, ""iterations"": 400, ""l2_leaf_reg"":...",67.527148,57.633044,476.620095,485.433317,0.032068,0.023268
1,regression_si,regression,SI,mae,nested,extra_trees,direct,111.338981,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m...",67.920799,57.670936,475.892696,484.578699,0.028192,0.027483
2,regression_si,regression,SI,mae,nested,random_forest,direct,238.516946,"{""max_depth"": null, ""max_features"": ""sqrt"", ""m...",68.016382,57.370337,476.379143,483.995173,0.023922,0.025946
3,regression_si,regression,SI,mae,nested,gradient_boosting,direct,235.692208,"{""learning_rate"": 0.1, ""max_depth"": 3, ""n_esti...",68.052133,57.554562,477.582162,485.574484,0.022142,0.018506
4,regression_si,regression,SI,mae,nested,svr,direct,11.414912,"{""C"": 1.0, ""epsilon"": 0.01, ""gamma"": ""scale""}",68.913082,57.192096,481.616150,485.730563,-0.004109,0.007739
5,regression_si,regression,SI,mae,nested,knn,direct,5.444663,"{""n_neighbors"": 21, ""p"": 1, ""weights"": ""distan...",69.377774,58.253947,478.077447,485.330999,0.024310,0.037332
6,regression_si,regression,SI,mae,nested,dummy,direct,0.465874,"{""strategy"": ""median""}",70.674322,57.306462,484.642740,486.113976,-0.029034,0.009768
7,regression_si,regression,SI,mae,nested,ridge,direct,0.993544,"{""alpha"": 100.0}",70.696331,56.408423,485.109729,483.148690,-0.067493,0.130074
8,regression_si,regression,SI,mae,nested,lasso,direct,15.479859,"{""alpha"": 0.1}",71.044037,57.127247,484.058711,486.145318,-0.018662,0.020482


,task_name,problem_type,target_column,primary_metric,evaluation_strategy,model_name,mode,fit_seconds,best_params_json,mae,mae_std,rmse,rmse_std,r2,r2_std
0,regression_si,regression,SI,mae,nested,extra_trees,indirect,219.480868,"{""cc50_component"": {""max_depth"": null, ""max_fe...",6.729125e+01,5.735437e+01,4.785548e+02,4.856427e+02,2.206342e-02,2.567825e-02
1,regression_si,regression,SI,mae,nested,random_forest,indirect,475.730031,"{""cc50_component"": {""max_depth"": null, ""max_fe...",6.770791e+01,5.753012e+01,4.781361e+02,4.861901e+02,2.918784e-02,3.664024e-02
2,regression_si,regression,SI,mae,nested,catboost,indirect,1601.896838,"{""cc50_component"": {""depth"": 6, ""iterations"": ...",6.797636e+01,5.757602e+01,4.799852e+02,4.862257e+02,1.702360e-02,3.141417e-02
3,regression_si,regression,SI,mae,nested,gradient_boosting,indirect,457.094882,"{""cc50_component"": {""learning_rate"": 0.1, ""max...",6.879816e+01,5.757303e+01,4.800713e+02,4.868931e+02,2.010994e-02,3.646404e-02
4,regression_si,regression,SI,mae,nested,knn,indirect,9.761202,"{""cc50_component"": {""n_neighbors"": 5, ""p"": 1, ...",7.037726e+01,5.783151e+01,4.801137e+02,4.865232e+02,1.369668e-02,3.514700e-02
5,regression_si,regression,SI,mae,nested,dummy,indirect,0.905867,"{""cc50_component"": {""strategy"": ""median""}, ""ic...",7.152377e+01,5.730590e+01,4.839034e+02,4.862643e+02,-2.151901e-02,5.169256e-03
6,regression_si,regression,SI,mae,nested,svr,indirect,22.992200,"{""cc50_component"": {""C"": 1.0, ""epsilon"": 0.01,...",2.815054e+06,5.630026e+06,3.990950e+07,7.981851e+07,-3.964675e+09,7.929351e+09
7,regression_si,regression,SI,mae,nested,ridge,indirect,1.949977,"{""cc50_component"": {""alpha"": 10.0}, ""ic50_comp...",3.090850e+06,3.845246e+06,3.834585e+07,5.030783e+07,-3.630954e+10,7.149546e+10
8,regression_si,regression,SI,mae,nested,lasso,indirect,29.377389,"{""cc50_component"": {""alpha"": 0.1}, ""ic50_compo...",3.642351e+07,5.621225e+07,5.132523e+08,7.908767e+08,-5.363395e+13,1.071995e+14


In [9]:
direct_importance = RESULTS_DIR / 'regression_si' / 'winner_feature_importance_direct.csv'
if direct_importance.exists():
    display(pd.read_csv(direct_importance).head(15))

for component_name in ('ic50_component', 'cc50_component'):
    component_path = RESULTS_DIR / 'regression_si' / f'winner_feature_importance_indirect_{component_name}.csv'
    if component_path.exists():
        print(component_name)
        display(pd.read_csv(component_path).head(15))

,feature,importance,abs_importance
0,FractionCSP3,3.529540,3.529540
1,VSA_EState8,3.219643,3.219643
2,VSA_EState6,2.704349,2.704349
3,SMR_VSA7,2.206567,2.206567
4,NHOHCount,1.945408,1.945408
5,BCUT2D_MRLOW,1.872402,1.872402
6,FpDensityMorgan3,1.842417,1.842417
7,NumSaturatedCarbocycles,1.815449,1.815449
8,AvgIpc,1.687163,1.687163
9,BCUT2D_MWLOW,1.674524,1.674524


ic50_component


,feature,importance,abs_importance
0,fr_NH2,0.045412,0.045412
1,NumSaturatedHeterocycles,0.037698,0.037698
2,NHOHCount,0.026620,0.026620
3,NumHDonors,0.017429,0.017429
4,NumSaturatedCarbocycles,0.017304,0.017304
5,PEOE_VSA7,0.016750,0.016750
6,fr_NH1,0.016670,0.016670
7,VSA_EState8,0.016519,0.016519
8,fr_Al_OH_noTert,0.015297,0.015297
9,EState_VSA8,0.015263,0.015263


cc50_component


,feature,importance,abs_importance
0,fr_NH2,0.031441,0.031441
1,NHOHCount,0.021558,0.021558
2,fr_allylic_oxid,0.017746,0.017746
3,Kappa3,0.017458,0.017458
4,PEOE_VSA6,0.016230,0.016230
5,BCUT2D_MWLOW,0.015910,0.015910
6,NumHDonors,0.014815,0.014815
7,MolLogP,0.013992,0.013992
8,PEOE_VSA7,0.013739,0.013739
9,Kappa2,0.012941,0.012941


## Итог

После запуска здесь остаётся полный, воспроизводимый срез по задаче: подготовка данных,
сравнительный запуск моделей, лидерборд и основные артефакты. Этого достаточно и для
защиты решения, и для быстрых повторных прогонов.